# Jerry GDELT 2015-2025 workflow

这不是展示版 notebook，而是 Jerry 的执行记录。按顺序运行，不要直接 Run All。每一个 gate 通过后再进入下一段。

当前最重要的事实：

- 不能把 `MakkaPakka update/scripts/ingest/06_spark_ingest_gdelt.py`、`07_spark_sentiment.py`、`11_spark_similarity.py` 原样串起来。它们的输入输出 schema 目前不兼容。
- AWS Open Data 的 GDELT mirror 已停止更新并截在 2019 年。必须从 GDELT 官方 HTTPS master list 选取 `.gkg.csv.zip`。
- GDELT 2.0 没覆盖 2015-01-28 会议；官方归档在 2025-06-15 至 2025-07-01 还有整段缺口。因此 lag -2 的理论最大样本是 86 次会议，不是 88。
- 当前 GDELT 的 `text` 实际是 themes、organizations、persons 拼接的 metadata，不是新闻正文。最终论文必须称为 GKG metadata-to-minutes lexical alignment，除非另行取得并验证文章正文。
- 当前 `vol_post` 不是论文声称的干净五日会后波动率；它混入了会前收益。这个问题会明显改变 headline coefficient，必须先修。
- 2015-2019 的 33 次会议已经被用于找 lag -2，应视为 discovery sample。新增的 2019-2025 有效 53 次会议应作为 temporal holdout，不能看完后再改主 lag、主筛选规则或主 outcome。
- 全量 rerun 必须写入新的 versioned S3 prefix，不能覆盖现有 reference artifacts。

结论先写清楚：Jerry 跑完数据并不等于只剩排版。跑完后还要通过构念、结果变量、holdout、multiple testing、样本选择、MPS、placebo、复现性和论文表述九类验收。

## 0. Jerry 打开什么、在哪里运行

先打开这个 notebook，再依次打开下面五个主文件。前三个负责 GDELT/TF-IDF，第四个负责 similarity，第五个负责正确市场 outcome。

1. `ingestion/06_ingest_gdelt.py`
2. `spark/08_spark_preprocess_tfidf.py`
3. `spark/09_spark_meeting_windows.py`
4. `MakkaPakka update/scripts/similarity/11_spark_similarity.py`
5. `analysis/12_lead_lag_volatility.py`

再打开三个下游文件，等云端结果回来以后修正并重跑：

6. `MakkaPakka update/scripts/analysis/00_build_panel.py`
7. `MakkaPakka update/scripts/analysis/12_main_regression.py`
8. `MakkaPakka update/scripts/analysis/13_identification.py`

不要把 `ingestion/03_ingest_fomc.py` 的 hard-coded meeting list 当真值。以 `MakkaPakka update/data_reference/meetings_all88.csv` 和已经核验过的 `fomc_minutes_raw.csv` 为准。

建议在 us-east-1 的长期 EC2/EMR Studio Jupyter 环境运行下载段，避免个人电脑承担数百 GiB 压缩流量。先在本地建立环境：

```bash
cd "/absolute/path/to/113 (including 112) FOMC project"
conda env create -f environment.yml
conda activate makkapakka
python -m ipykernel install --user --name makkapakka --display-name "Python (makkapakka)"
jupyter lab
```

在 Jupyter 中选择 `Python (makkapakka)`。不要把 `.env`、AWS credentials 或 API keys 上传到 S3。

三天技术 pilot 可以在本地完成。见 [本地采集和 Spark 试跑命令](../ingestion/README.md#local-three-day-pilot)。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile

import boto3
import numpy as np
import pandas as pd
import requests

ROOT = Path(os.environ.get("MP_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
if ROOT.name == "MakkaPakka update":
    ROOT = ROOT.parent

UPDATE = ROOT / "MakkaPakka update"

assert ROOT.is_dir(), f"Project directory missing: {ROOT}"
assert UPDATE.is_dir(), f"Update directory missing: {UPDATE}"

print("ROOT:", ROOT)
print("UPDATE:", UPDATE)

RUN_ID = os.environ.get("MP_RUN_ID", "gdelt_2015_2025_q1_v1")
RUN_DIR = UPDATE / "handoff" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
PAD_DAYS = int(os.environ.get("MP_PAD_DAYS", "7"))
MAX_DOWNLOAD_GIB = float(os.environ.get("MP_MAX_DOWNLOAD_GIB", "1100"))
ALLOW_CLOUD_WRITE = os.environ.get("MP_ALLOW_CLOUD_WRITE", "0") == "1"
ALLOW_FULL_RUN = os.environ.get("MP_ALLOW_FULL_RUN", "0") == "1"
ALLOW_EMR = os.environ.get("MP_ALLOW_EMR", "0") == "1"

assert (ROOT / "ingestion" / "06_ingest_gdelt.py").exists()
assert (UPDATE / "data_reference" / "meetings_all88.csv").exists()
assert PAD_DAYS in (3, 7)
print(ROOT)
print(RUN_DIR)
print(PAD_DAYS)

In [ ]:
def run(args, cwd=ROOT, capture=False):
    return subprocess.run(
        [str(x) for x in args],
        cwd=cwd,
        check=True,
        text=True,
        capture_output=capture,
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def s3_count(client, bucket, prefix, suffix=""):
    total = 0
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        total += sum(x["Key"].endswith(suffix) for x in page.get("Contents", []))
    return total

## 1. 先留可复现记录

当前项目根目录不是 Git repository，因此至少要保存每个关键脚本的 SHA-256、Python package freeze、Spark/Java/AWS CLI 版本和之后的 EMR cluster description。若团队允许，正式跑之前应把代码放进 private Git repository 并打 tag。

In [ ]:
CODE_FILES = [
    ROOT / "ingestion" / "06_ingest_gdelt.py",
    ROOT / "ingestion" / "gdelt_filter_rules_v1.json",
    ROOT / "spark" / "08_spark_preprocess_tfidf.py",
    ROOT / "spark" / "09_spark_meeting_windows.py",
    UPDATE / "scripts" / "similarity" / "11_spark_similarity.py",
    ROOT / "analysis" / "12_lead_lag_volatility.py",
    ROOT / "scripts" / "analysis" / "temporal_validation.py",
    UPDATE / "scripts" / "analysis" / "00_build_panel.py",
    UPDATE / "scripts" / "analysis" / "12_main_regression.py",
    UPDATE / "scripts" / "analysis" / "13_identification.py",
    UPDATE / "Jerry_GDELT_2015_2025_workflow.ipynb",
    UPDATE / "analysis_protocol_extension.md",
]

code_manifest = {str(path.relative_to(ROOT)): sha256_file(path) for path in CODE_FILES}
with open(RUN_DIR / "code_manifest_before.json", "w", encoding="utf-8") as handle:
    json.dump(code_manifest, handle, indent=2, sort_keys=True)

freeze = run([sys.executable, "-m", "pip", "freeze"], capture=True).stdout
(RUN_DIR / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
print(sys.version)
print(sys.executable)
print(json.dumps(code_manifest, indent=2, sort_keys=True))

In [ ]:
for name in ["aws", "java", "spark-submit"]:
    print(name, shutil.which(name))

run([sys.executable, "-c", "import pandas,numpy,pyarrow,boto3,requests; print('python-dependencies-ok')"])



`aws` 必须存在，因为 notebook 用它同步 S3。`java` 与本地 `spark-submit` 只做记录；真正 Spark stages 在 EMR 上执行时，本地可以没有它们。不要在 notebook 中临时安装一套没有记录的环境；缺 Python 依赖时先修 `environment.yml`，重建 kernel，再从这里继续。

## 2. 核验 canonical inputs

88 是会议和 minutes 的数量，不是最终 GDELT lag -2 可用数量。2020 的两个特殊日期必须是 2020-03-15 和 2020-04-29。

In [ ]:
MEETING_FILE = UPDATE / "data_reference" / "meetings_all88.csv"
MINUTES_FILE = UPDATE / "data_reference" / "fomc_minutes_raw.csv"
SPX_FILE = UPDATE / "data_reference" / "market" / "SPX_daily_2014_2025.csv"
VIX_FILE = UPDATE / "data_reference" / "market" / "VIX_2014_2025.csv"
MPS_FILE = UPDATE / "data_reference" / "market" / "mps_fomc.csv"

expected_hashes = {
    "meetings_all88.csv": "48ece57c5a810c5f1c9093c9971f8c3acad932b50a828748cc0e5806f680432b",
    "fomc_minutes_raw.csv": "c303fab1bcb8cf76f11af95ee76111aa78cc1a7f6e43e1e21203fd0b202a12b0",
    "SPX_daily_2014_2025.csv": "0d11e8a27df5e31bda31a869b3e8fa9db52baef15f3f3d3180412627df313f1f",
    "VIX_2014_2025.csv": "ff11d8e6eaa1dcd29d622501a44e7ae29100f6b0fd1111fadfbf7b9e9e7b8c03",
    "mps_fomc.csv": "f11038dd7045f5981aa086879c42998d2d21f9019daf1474fb5c1c15b97af96d",
}

for path in [MEETING_FILE, MINUTES_FILE, SPX_FILE, VIX_FILE, MPS_FILE]:
    actual = sha256_file(path)
    print(path.name, actual)
    assert actual == expected_hashes[path.name]

meetings = pd.read_csv(MEETING_FILE, parse_dates=["meeting_date"])
minutes = pd.read_csv(MINUTES_FILE, dtype={"date": "string"})
minutes["meeting_date"] = pd.to_datetime(minutes["date"], format="%Y%m%d")

assert len(meetings) == 88
assert meetings["meeting_date"].nunique() == 88
assert len(minutes) == 88
assert minutes["meeting_date"].nunique() == 88
assert set(meetings["meeting_date"]) == set(minutes["meeting_date"])
assert minutes["text"].str.len().min() > 500

dates_2020 = set(meetings.loc[meetings["meeting_date"].dt.year == 2020, "meeting_date"].dt.strftime("%Y-%m-%d"))
assert "2020-03-15" in dates_2020
assert "2020-04-29" in dates_2020
assert "2020-03-11" not in dates_2020
assert "2020-04-15" not in dates_2020
print(meetings["meeting_date"].min(), meetings["meeting_date"].max())
print(minutes["text"].str.len().describe())

In [ ]:
spx = pd.read_csv(SPX_FILE, parse_dates=["date"]).sort_values("date")
vix = pd.read_csv(VIX_FILE, parse_dates=["observation_date"]).sort_values("observation_date")
mps = pd.read_csv(MPS_FILE, parse_dates=["Date"]).sort_values("Date")
current_panel = pd.read_parquet(UPDATE / "data_reference" / "per_meeting.parquet")
current_panel["meeting_date"] = pd.to_datetime(current_panel["meeting_date"])

print("spx", spx["date"].min(), spx["date"].max())
print("vix", vix["observation_date"].min(), vix["observation_date"].max())
print("mps", mps["Date"].min(), mps["Date"].max())
print("gdelt-current", current_panel.loc[current_panel["source"] == "gdelt", "meeting_date"].min(), current_panel.loc[current_panel["source"] == "gdelt", "meeting_date"].max())

assert spx["date"].max() >= pd.Timestamp("2025-12-30")
assert vix["observation_date"].max() >= pd.Timestamp("2025-12-30")
assert mps["Date"].max() == pd.Timestamp("2023-12-13")

上一个 cell 应清楚显示：市场价格到 2025 年底，旧 MPS 只到 2023-12-13，当前 GDELT 只到 2019-03-20。MPS 缺口不能通过全局 `dropna` 静默缩短所有回归样本。主规格 1-4 应使用完整可用样本；含 MPS 的规格要单独报告较短样本，并给出相同样本上的不含 MPS bridge column。最好改用 SF Fed USMPD 的一致 statement-surprise series 覆盖整个时期，不要把不同方法的两段 series 静默拼接。

下载并冻结当前 SF Fed USMPD 与官方 monetary-policy-surprises bundle。不要覆盖旧 Bauer-Swanson 文件。下载日期、URL、SHA-256、sheet/file names 都进入 run manifest。之后选择 meeting-statement 或完整 monetary-event surprise 时，要在 protocol 中只选一个 primary definition，并用同一方法覆盖全时期。

In [ ]:
USMPD_URL = "https://www.frbsf.org/wp-content/uploads/USMPD.xlsx"
SURPRISE_URL = "https://www.frbsf.org/wp-content/uploads/monetary-policy-surprises.zip"
USMPD_FILE = RUN_DIR / "USMPD.xlsx"
SURPRISE_FILE = RUN_DIR / "monetary-policy-surprises.zip"

for url, path in [(USMPD_URL, USMPD_FILE), (SURPRISE_URL, SURPRISE_FILE)]:
    response = requests.get(url, timeout=(30, 180))
    response.raise_for_status()
    assert response.content[:2] == b"PK"
    path.write_bytes(response.content)
    print(path.name, len(response.content), sha256_file(path))

print(pd.ExcelFile(USMPD_FILE).sheet_names)
with zipfile.ZipFile(SURPRISE_FILE) as archive:
    print(archive.namelist())

## 3. 先确认 outcome bug，不要在旧结果上继续写结论

旧 `analysis/12_lead_lag_volatility.py` 先算 rolling 5-day standard deviation，再对五个 rolling values 求平均。这样第一个所谓 post value 会使用会前 return。最终代码必须从冻结的 SPX CSV 直接选出不重叠的 return dates。主文同时报告 event-inclusive 与 strictly-post 两种定义，并保存每次会议实际使用的交易日。

In [ ]:
spx = spx.copy()
spx["logret"] = np.log(spx["spx"]).diff()

def volatility_windows(frame, meeting, window=5):
    meeting = pd.Timestamp(meeting)
    pre = frame.loc[frame["date"] < meeting].tail(window)
    strict = frame.loc[frame["date"] > meeting].head(window)
    event = frame.loc[frame["date"] >= meeting].head(window)
    assert len(pre) == window
    assert len(strict) == window
    assert len(event) == window
    assert set(pre["date"]).isdisjoint(set(strict["date"]))
    scale = np.sqrt(252.0)
    return {
        "meeting_date": meeting,
        "vol_pre_5": pre["logret"].std(ddof=1) * scale,
        "vol_post_strict_5": strict["logret"].std(ddof=1) * scale,
        "vol_post_event_5": event["logret"].std(ddof=1) * scale,
        "pre_dates": "|".join(pre["date"].dt.strftime("%Y-%m-%d")),
        "strict_dates": "|".join(strict["date"].dt.strftime("%Y-%m-%d")),
        "event_dates": "|".join(event["date"].dt.strftime("%Y-%m-%d")),
    }

strict_vol = pd.DataFrame([volatility_windows(spx, date) for date in meetings["meeting_date"]])
strict_vol.to_csv(RUN_DIR / "volatility_window_audit.csv", index=False)
baseline = current_panel[(current_panel["source"] == "gdelt") & (current_panel["lag_days"] == -2)].copy()
baseline = baseline.drop_duplicates("meeting_date").merge(strict_vol, on="meeting_date", how="inner")
print("legacy", baseline["cosine_sim"].corr(baseline["vol_post"]))
print("strict", baseline["cosine_sim"].corr(baseline["vol_post_strict_5"]))
print("event", baseline["cosine_sim"].corr(baseline["vol_post_event_5"]))

当前 33 次会议上，legacy correlation 约为 0.565，strict five-return correlation 约为 0.366。这个差异足以改变论文 headline，所以旧摘要数字不能继续被当作最终结果。

## 4. 从官方 GDELT master list 生成精确下载计划

`PAD_DAYS=7` 对应原论文实际查看过的 -7 到 +7 lag family，成本最高但能做完整的 max-statistic multiple-testing correction。若只为了冻结的 holdout lag -2，可以把 `MP_PAD_DAYS` 设为 3；但若正文保留 -7 到 +7 的 lead-lag figure，一区标准下最终仍应补齐 7 日窗口。下面按 America/New_York 把 UTC shard timestamp 映射到本地 calendar day。

In [ ]:
MASTER_URL = "https://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
PLAN_FILE = RUN_DIR / "gdelt_download_plan.csv"
PLAN_META_FILE = RUN_DIR / "gdelt_download_plan_metadata.json"
target_days = {
    (meeting + pd.Timedelta(days=offset)).date()
    for meeting in meetings["meeting_date"]
    for offset in range(-PAD_DAYS, PAD_DAYS + 1)
}
eastern = ZoneInfo("America/New_York")
master_digest = hashlib.sha256()
rows = []

with requests.get(MASTER_URL, stream=True, timeout=(30, 900)) as response:
    response.raise_for_status()
    for raw in response.iter_lines():
        master_digest.update(raw + b"\n")
        parts = raw.decode("ascii").split()
        if len(parts) != 3:
            continue
        match = re.search(r"/(\d{14})\.gkg\.csv\.zip$", parts[2], re.I)
        if match is None:
            continue
        stamp = match.group(1)
        instant = datetime.strptime(stamp, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)
        eastern_date = instant.astimezone(eastern).date()
        if eastern_date not in target_days:
            continue
        rows.append({
            "stamp": stamp,
            "eastern_date": eastern_date.isoformat(),
            "compressed_bytes": int(parts[0]),
            "md5": parts[1],
            "url": parts[2].replace("http://", "https://", 1),
        })

plan = pd.DataFrame(rows).drop_duplicates("url").sort_values("stamp").reset_index(drop=True)
planned_gib = plan["compressed_bytes"].sum() / 1024 ** 3
assert not plan.empty
assert plan["stamp"].is_unique
assert plan["url"].str.endswith(".gkg.csv.zip").all()
assert planned_gib <= MAX_DOWNLOAD_GIB
plan.to_csv(PLAN_FILE, index=False)
plan_meta = {
    "master_url": MASTER_URL,
    "master_sha256": master_digest.hexdigest(),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "pad_days": PAD_DAYS,
    "selected_shards": int(len(plan)),
    "planned_gib": float(planned_gib),
    "plan_sha256": sha256_file(PLAN_FILE),
}
with open(PLAN_META_FILE, "w", encoding="utf-8") as handle:
    json.dump(plan_meta, handle, indent=2, sort_keys=True)
print(json.dumps(plan_meta, indent=2, sort_keys=True))

In [ ]:
available_days = set(pd.to_datetime(plan["eastern_date"]).dt.date)
coverage = meetings.copy()
coverage["lag_m2_day"] = coverage["meeting_date"] - pd.Timedelta(days=2)
coverage["lag_m2_available"] = coverage["lag_m2_day"].dt.date.isin(available_days)
missing = coverage.loc[~coverage["lag_m2_available"], "meeting_date"].dt.strftime("%Y-%m-%d").tolist()
assert missing == ["2015-01-28", "2025-06-18"]
assert int(coverage["lag_m2_available"].sum()) == 86

discovery = coverage[coverage["lag_m2_available"] & (coverage["meeting_date"] <= pd.Timestamp("2019-03-20"))]
validation = coverage[coverage["lag_m2_available"] & (coverage["meeting_date"] >= pd.Timestamp("2019-05-01"))]
assert len(discovery) == 33
assert len(validation) == 53
coverage.to_csv(RUN_DIR / "meeting_archive_coverage.csv", index=False)
print("missing", missing)
print("discovery", len(discovery))
print("validation", len(validation))
print("pooled", int(coverage["lag_m2_available"].sum()))

## 5. 全量运行前必须冻结 extension protocol

在看 2019-2025 similarity 与 regression 结果前，共同确认已经附上的 `MakkaPakka update/analysis_protocol_extension.md` 草案。它不能称为 retrospective preregistration；准确名称是 prospective temporal-holdout protocol。Doris 和 Jerry 签名并填写 UTC freeze time 后，才把第一行从 `STATUS: DRAFT` 改成 `STATUS: FROZEN`。至少确认：

- discovery: 2015-03-18 至 2019-03-20，33 meetings
- holdout: 2019-05-01 至 2025-12-10，因官方 gap 排除 2025-06-18，53 meetings
- primary lag: -2
- primary corpus: direct Fed filter
- robustness corpus: original broad macro filter
- primary representation: GKG themes, organizations, persons metadata
- primary weighting: domain-equal weighting；article weighting 作为 robustness
- primary outcome: strict five-trading-day realized volatility，event-inclusive 作为 robustness
- pre-period volatility 的精确交易日定义
- 主 controls、缺失处理、双侧检验和标准误
- 先单独报告 untouched holdout，再报告 pooled sample
- 对原始 -7 至 +7 search family 做 max-|t| permutation correction
- block/circular permutation 与 time-aware bootstrap
- 不做 media-to-market causal claim
- 不因 holdout 结果改变 filter、lag 或 outcome


In [ ]:
PROTOCOL = UPDATE / "analysis_protocol_extension.md"
assert PROTOCOL.exists()
assert PROTOCOL.stat().st_size > 1500
protocol_text = PROTOCOL.read_text(encoding="utf-8").lower()
assert protocol_text.startswith("status: frozen")
assert "approved by doris: pending" not in protocol_text
assert "approved by jerry: pending" not in protocol_text
for token in ["discovery", "holdout", "lag -2", "strict", "max", "direct-fed"]:
    assert token in protocol_text
print(sha256_file(PROTOCOL))

## 6. Jerry 必须先改的文件

### `ingestion/06_ingest_gdelt.py`

本地 pilot 用 `--local-root` 保留相同的分片目录和清单。ZIP 校验后，清单把无效 UTF-8 或字段结构不完整的行计入 `malformed_rows`。没有有效行的分片判为失败。

删掉对 stale `gdelt-open-data` mirror 的 discovery/download 依赖。新接口统一为 `--bucket --plan --out-prefix --manifest-prefix --workers --resume`，可选 `--max-shards`。实现要求：

- 只读 notebook 生成的 official HTTPS plan。
- 校验 HTTP status、Content-Length 和 official MD5。
- 流式下载和解压；失败必须是 failed，不能变成 empty success。
- 每个 shard 写一个 manifest JSON；即使零匹配也写 `empty` completion。
- 每次结束时把 shard records 汇总成 `run_manifest.jsonl`，供 QC；resume 仍以每个 shard JSON 为准。
- 保存 UTC shard timestamp 和 Eastern calendar date。
- 输出字段至少包含 `source, doc_id, doc_date, doc_time_utc, source_domain, url, themes, organizations, persons, gkg_tone, metadata_text, representation_type, direct_fed, broad_macro, shard_stamp, filter_version`。
- `representation_type` 明确写成 `gkg_themes_orgs_persons`。
- 新旧 filter 不得写入同一 prefix。
- domain matching 只能 exact/suffix，不得使用 `ap`、`ft`、`inc` 之类 substring。

建议直接使用下面两个简单函数：

```python
def normalize_domain(value):
    value = (value or "").strip().lower().split(":")[0]
    if value.startswith("www."):
        value = value[4:]
    return value

def allowed_domain(value, allowed):
    value = normalize_domain(value)
    return any(value == item or value.endswith("." + item) for item in allowed)
```

把 outlet allowlist 和 direct/broad relevance rules 放进 versioned JSON，并把 hash 写进 run manifest。`direct_fed` 必须要求 Federal Reserve/FOMC entity、URL 或 named official 与 monetary-policy context；旧 broad themes 只能作为 robustness。

### `spark/08_spark_preprocess_tfidf.py`

版本化入口读取显式输入、输出和模型路径。GDELT 只读 `broad_macro` 分区，再按 `direct_fed` 标志区分语料。按来源和 document ID 去重，保留域名、native tone、语料标志和 representation。

TF-IDF 使用 discovery 期的 direct-Fed 和 FOMC minutes 拟合，范围是 2015-03-15 至 2019-03-23。使用同一份模型变换各期数据，向量先做 L2 归一化。模型和拟合样本数、cutoff、filter hash 与脚本 hash 写入指定模型目录。

### `spark/09_spark_meeting_windows.py`

通过 `--tfidf --meetings --output --max-lag` 生成会议窗口。先按日聚合文章向量和域名向量，再与会议日历连接。输出 article 和 domain-equal 两种权重，以及 `n_docs`、`n_urls`、`n_domains` 和 `mean_native_tone`。


### `MakkaPakka update/scripts/similarity/11_spark_similarity.py`

版本化入口用 `--windows` 和 `--tfidf` 接受：

- media windows: `meeting_date, lag_days, source, corpus, weighting, mean_features, n_docs, n_domains`
- FOMC vectors: `doc_date, source, features`

将 FOMC `doc_date` rename 为 `meeting_date`，直接对已经聚合的 window vector 算 cosine。输出保留 counts、corpus、weighting、model cutoff 和 representation version。

### Sentiment

GDELT 主 tone 应使用 GKG 自带的 V2Tone，因为它来自原始 article processing。FOMC minutes 继续使用 LM 和 FinBERT。不要把对 metadata 的自建 hawkish/dovish keyword count 称为 article sentiment。若仍保留自建 lexicon，Spark 和 Numba 必须读取同一个冻结文件，并用 fixture test 证明结果一致。

### `analysis/12_lead_lag_volatility.py`

- 删除运行时 `yfinance`。
- 只读已冻结并有 hash 的 SPX CSV。
- 删除 rolling-SD 再平均的旧逻辑。
- 生成 `vol_pre_w, vol_post_strict_w, vol_post_event_w`，w 为 3、5、7、10。
- 保存每个 window 的真实 trading dates，并 assert pre/post 不交叉。
- alternative-window regressions 必须让 `vol_post_w` 配对 `vol_pre_w`，不能总用 legacy `vol_pre`。

### `00_build_panel.py` 与 `12_main_regression.py`

- 不再信任旧 parquet 的 `vol_pre`、`vol_post`。
- 每一个 regression specification 自己做 complete-case selection。删除全表先按 `mps_orth` dropna 的逻辑。
- 分开报告 discovery、untouched holdout、pooled。
- 完整样本 columns 与 MPS-restricted columns 分开，并给 same-sample bridge。
- 报标准化 beta、partial R-squared、economic magnitude、confidence interval。
- 不把控制后不显著解释成证明零效应；若要主张实质零效应，先预设 equivalence bounds 再做 equivalence test。

## 7. 修改后的代码 gate

Jerry 完成上述 edits 后运行。任何一个 cell 失败都不要开始数百 GiB 的 full acquisition。项目还需要新增 `tests/test_gdelt_http.py` 和 `tests/test_volatility_windows.py`。

In [ ]:
# Rerun for Validation
import subprocess
import sys

result = subprocess.run(
    [sys.executable, str(ROOT / "ingestion" / "06_ingest_gdelt.py"), "--help"],
    capture_output=True,
    text=True
)

required = [
    "--bucket", "--plan", "--out-prefix",
    "--manifest-prefix", "--workers", "--resume"
]

print("Missing:", [x for x in required if x not in result.stdout])
print("Exit code:", result.returncode)

In [ ]:
INGEST = ROOT / "ingestion" / "06_ingest_gdelt.py"
PREPROCESS = ROOT / "spark" / "08_spark_preprocess_tfidf.py"
WINDOWS = ROOT / "spark" / "09_spark_meeting_windows.py"
SIMILARITY = UPDATE / "scripts" / "similarity" / "11_spark_similarity.py"
MARKET = ROOT / "analysis" / "12_lead_lag_volatility.py"
TEST_GDELT = ROOT / "tests" / "test_gdelt_http.py"
TEST_VOL = ROOT / "tests" / "test_volatility_windows.py"

for path in [INGEST, PREPROCESS, WINDOWS, SIMILARITY, MARKET]:
    run([sys.executable, "-m", "py_compile", path])
    print("syntax-ok", path)

ingest_help = run([sys.executable, INGEST, "--help"], capture=True).stdout
for flag in ["--bucket", "--plan", "--out-prefix", "--manifest-prefix", "--workers", "--resume"]:
    assert flag in ingest_help

assert TEST_GDELT.exists()
assert TEST_VOL.exists()
run([sys.executable, "-m", "pytest", "-q", TEST_GDELT])
run([sys.executable, "-m", "pytest", "-q", TEST_VOL])

In [ ]:
print("=== Current ingestion script help ===")
print(ingest_help)

required_flags = [
    "--bucket",
    "--plan",
    "--out-prefix",
    "--manifest-prefix",
    "--workers",
    "--resume"
]

missing_flags = [
    flag for flag in required_flags
    if flag not in ingest_help
]

print("\nMissing CLI arguments:", missing_flags)

最低测试必须覆盖：27-column GKG parse、ZIP/MD5 failure、network failure 不能记成 empty、integer FOMC date、exact/suffix domain matching、direct/broad flags、duplicate handling、zero-match completion manifest、pre/post dates 不交叉、2015-01 和 2025-06 archive gaps、lag -2 最大 n=86。

## 8. AWS preflight 与独立 staging bucket

在 shell 中设置变量，不写入 notebook：

```bash
export AWS_DEFAULT_REGION=us-east-1
export MP_AWS_ACCOUNT_ID=123456789012
export MP_SOURCE_BUCKET=existing-source-bucket
export MP_RUN_BUCKET=new-versioned-run-bucket
export MP_RUN_ID=gdelt_2015_2025_q1_v1
export MP_ALLOW_CLOUD_WRITE=0
```

`SOURCE_BUCKET` 与 `RUN_BUCKET` 必须不同。RUN_BUCKET 需要 versioning、server-side encryption、public-access block、billing alert 和 S3 log prefix。不要运行旧 `spark/launch_spark_cluster.py`；它使用旧 EMR release，并曾设计成打开 public SSH。

In [ ]:
import boto3
import os

profile_name = os.environ.get("MP_AWS_PROFILE") or os.environ.get("AWS_PROFILE")
session = boto3.Session(
    profile_name=profile_name,
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)
print("AWS account:", session.client("sts").get_caller_identity()["Account"])
print("Configured run bucket:", repr(os.environ.get("MP_RUN_BUCKET")))


Set the AWS profile, region and run bucket in the shell before starting the notebook. Keep the cloud-write, full-run and EMR flags disabled during preflight.

In [ ]:
import os
import boto3

REGION = os.environ.get(
    "AWS_DEFAULT_REGION", "us-east-1"
)

AWS_PROFILE = os.environ.get("MP_AWS_PROFILE") or os.environ.get("AWS_PROFILE")

RUN_BUCKET = os.environ.get(
    "MP_RUN_BUCKET", ""
).strip()

SOURCE_BUCKET = os.environ.get(
    "MP_SOURCE_BUCKET", ""
).strip()

LOCAL_SOURCE_DIR = Path(
    os.environ.get(
        "MP_LOCAL_SOURCE_DIR",
        str(ROOT / "source_data")
    )
)

# Validate the named AWS profile
available_profiles = boto3.Session().available_profiles

if AWS_PROFILE and AWS_PROFILE not in available_profiles:
    raise RuntimeError(
        f"Profile {AWS_PROFILE} not found. "
        f"Available profiles: {available_profiles}"
    )

# Use the same profile for Boto3 and AWS CLI
if AWS_PROFILE:
    os.environ["AWS_PROFILE"] = AWS_PROFILE

session = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=REGION
)

sts = session.client("sts")
s3 = session.client("s3")

# Read the authenticated AWS account
identity = sts.get_caller_identity()

print("AWS Account:", identity["Account"])
print("AWS Identity:", identity["Arn"])
print("Region:", REGION)

if identity["Arn"].endswith(":root"):
    print(
        "WARNING: Using AWS root identity. "
        "Use a separate IAM identity for routine cloud operations."
    )

# Optional expected account verification
EXPECTED_ACCOUNT = os.environ.get(
    "MP_AWS_ACCOUNT_ID", ""
).strip()

if EXPECTED_ACCOUNT:
    assert identity["Account"] == EXPECTED_ACCOUNT, (
        "Authenticated AWS account does not match "
        "MP_AWS_ACCOUNT_ID"
    )

# Validate the run bucket without creating it
if not RUN_BUCKET:
    raise RuntimeError(
        "MP_RUN_BUCKET is missing. Set your existing "
        "S3 bucket name in your environment."
    )

if SOURCE_BUCKET and SOURCE_BUCKET == RUN_BUCKET:
    raise RuntimeError(
        "Source and run buckets must be different."
    )

s3.head_bucket(Bucket=RUN_BUCKET)
print("Run bucket accessible:", RUN_BUCKET)

# Original research S3 bucket is optional here
if SOURCE_BUCKET:
    s3.head_bucket(Bucket=SOURCE_BUCKET)
    print("Source bucket accessible:", SOURCE_BUCKET)
else:
    print(
        "No source bucket configured. "
        "Local NYT/Guardian staging will be needed."
    )

# Safety check for all subsequent writes
def require_cloud_write():
    if os.environ.get("MP_ALLOW_CLOUD_WRITE") != "1":
        raise RuntimeError(
            "Cloud writes are disabled. "
            "Enable MP_ALLOW_CLOUD_WRITE only after "
            "confirming the AWS account and bucket."
        )

    expected = os.environ.get(
        "MP_AWS_ACCOUNT_ID", ""
    ).strip()

    if expected != identity["Account"]:
        raise RuntimeError(
            "Set MP_AWS_ACCOUNT_ID to the "
            "verified AWS account number first."
        )

    if not RUN_BUCKET or (
        SOURCE_BUCKET and SOURCE_BUCKET == RUN_BUCKET
    ):
        raise RuntimeError(
            "Unsafe S3 bucket configuration."
        )

print("AWS read-only preflight passed.")


In [ ]:
s3.put_bucket_versioning(Bucket=RUN_BUCKET, VersioningConfiguration={"Status": "Enabled"})
s3.put_bucket_encryption(
    Bucket=RUN_BUCKET,
    ServerSideEncryptionConfiguration={
        "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]
    },
)
s3.put_public_access_block(
    Bucket=RUN_BUCKET,
    PublicAccessBlockConfiguration={
        "BlockPublicAcls": True,
        "IgnorePublicAcls": True,
        "BlockPublicPolicy": True,
        "RestrictPublicBuckets": True,
    },
)
print(s3.get_bucket_versioning(Bucket=RUN_BUCKET))
print(s3.get_bucket_encryption(Bucket=RUN_BUCKET))

## 9. 上传 canonical inputs、plan、protocol 和 patched code

FOMC 直接用已核验的 88 份 local minutes，不要重跑旧 hard-coded scraper。NYT/Guardian 从原 source bucket 复制到 staging bucket。上传前后都记 counts。

In [ ]:
for row in minutes.itertuples(index=False):
    stamp = str(row.date)
    date = pd.to_datetime(stamp, format="%Y%m%d").strftime("%Y-%m-%d")
    body = {
        "source": "fomc",
        "doc_id": f"fomc_{stamp}",
        "doc_date": date,
        "url": f"https://www.federalreserve.gov/monetarypolicy/fomcminutes{stamp}.htm",
        "text": row.text,
    }
    s3.put_object(
        Bucket=RUN_BUCKET,
        Key=f"runs/{RUN_ID}/raw/fomc/{date}/fomc_{stamp}.json",
        Body=json.dumps(body).encode("utf-8"),
        ContentType="application/json",
    )

assert s3_count(s3, RUN_BUCKET, f"runs/{RUN_ID}/raw/fomc/", ".json") == 88
print("fomc-upload-ok")

In [ ]:
for source in ["nyt", "guardian"]:
    run([
        "aws",
        "s3",
        "sync",
        f"s3://{SOURCE_BUCKET}/raw/{source}/",
        f"s3://{RUN_BUCKET}/runs/{RUN_ID}/raw/{source}/",
        "--only-show-errors",
    ])

uploads = {
    MEETING_FILE: f"runs/{RUN_ID}/ref/meetings_all88.csv",
    SPX_FILE: f"runs/{RUN_ID}/ref/SPX_daily_2014_2025.csv",
    VIX_FILE: f"runs/{RUN_ID}/ref/VIX_2014_2025.csv",
    MPS_FILE: f"runs/{RUN_ID}/ref/mps_fomc_legacy.csv",
    USMPD_FILE: f"runs/{RUN_ID}/ref/USMPD.xlsx",
    SURPRISE_FILE: f"runs/{RUN_ID}/ref/monetary-policy-surprises.zip",
    PLAN_FILE: f"runs/{RUN_ID}/manifests/gdelt_download_plan.csv",
    PLAN_META_FILE: f"runs/{RUN_ID}/manifests/gdelt_download_plan_metadata.json",
    PROTOCOL: f"runs/{RUN_ID}/manifests/analysis_protocol_extension.md",
}

for local_path, key in uploads.items():
    s3.upload_file(str(local_path), RUN_BUCKET, key)

for path in CODE_FILES:
    key = f"runs/{RUN_ID}/code/{path.name}"
    s3.upload_file(str(path), RUN_BUCKET, key)

print("nyt", s3_count(s3, RUN_BUCKET, f"runs/{RUN_ID}/raw/nyt/"))
print("guardian", s3_count(s3, RUN_BUCKET, f"runs/{RUN_ID}/raw/guardian/"))

## 10. Pilot acquisition

先选 2019、2020、2025 三个时期的完整 calendar days。pilot 必须验证 ZIP、MD5、schema、timezone、domain matching、direct/broad flags、native tone、empty completion 和 resume。没有 pilot pass，不得运行 full acquisition。

In [ ]:
pilot_days = {"2019-03-18", "2020-03-13", "2025-12-08"}
pilot_plan = plan[plan["eastern_date"].isin(pilot_days)].copy()
PILOT_PLAN_FILE = RUN_DIR / "gdelt_pilot_plan.csv"
pilot_plan.to_csv(PILOT_PLAN_FILE, index=False)
assert set(pilot_plan["eastern_date"].unique()) == pilot_days
print(pilot_plan.groupby("eastern_date").size())

pilot_cmd = [
    sys.executable,
    INGEST,
    "--bucket",
    RUN_BUCKET,
    "--plan",
    PILOT_PLAN_FILE,
    "--out-prefix",
    f"pilot/{RUN_ID}/raw/gdelt",
    "--manifest-prefix",
    f"pilot/{RUN_ID}/manifests/gdelt",
    "--workers",
    "4",
    "--resume",
]
print(" ".join(str(x) for x in pilot_cmd))
run(pilot_cmd)

In [ ]:
PILOT_MANIFEST_FILE = RUN_DIR / "gdelt_pilot_manifest.jsonl"
pilot_manifest_key = f"pilot/{RUN_ID}/manifests/gdelt/run_manifest.jsonl"
s3.download_file(RUN_BUCKET, pilot_manifest_key, str(PILOT_MANIFEST_FILE))
pilot_manifest = pd.read_json(PILOT_MANIFEST_FILE, lines=True)
required_manifest = {"stamp", "url", "status", "md5_ok", "parsed_rows", "malformed_rows", "matched_rows", "filter_version"}
assert required_manifest.issubset(pilot_manifest.columns)
assert len(pilot_manifest) == len(pilot_plan)
assert pilot_manifest["status"].isin(["success", "empty"]).all()
assert pilot_manifest["md5_ok"].all()
assert (pilot_manifest["parsed_rows"] > 0).all()
assert pilot_manifest["stamp"].nunique() == len(pilot_plan)
print(pilot_manifest["status"].value_counts())
print("malformed GKG rows", int(pilot_manifest["malformed_rows"].sum()))
print(pilot_manifest[["parsed_rows", "malformed_rows", "matched_rows"]].describe())

除了自动 gate，Jerry 必须人工抽看 pilot 至少 100 条 direct-Fed 和 100 条 broad-macro records。记录 false positive 类型、domain distribution、重复 URL 比例。direct corpus 应是 broad corpus 的可解释子集。若 false positives 明显，修改 filter_version，重新跑 pilot，并创建新 run prefix；不能在旧 prefix 上混合。

## 11. Full acquisition

只有 protocol、tests、pilot、预算和 staging checks 都通过后，才设置 `MP_ALLOW_FULL_RUN=1`，重启 kernel 并重新运行必要 cells。full run 应在同 region 的长期实例中执行；脚本必须支持 manifest-based resume。

In [ ]:
assert ALLOW_FULL_RUN
assert ALLOW_CLOUD_WRITE
assert SOURCE_BUCKET != RUN_BUCKET
assert PROTOCOL.exists()
assert len(pilot_manifest) == len(pilot_plan)
assert pilot_manifest["status"].isin(["success", "empty"]).all()
assert planned_gib <= MAX_DOWNLOAD_GIB

full_cmd = [
    sys.executable,
    INGEST,
    "--bucket",
    RUN_BUCKET,
    "--plan",
    PLAN_FILE,
    "--out-prefix",
    f"runs/{RUN_ID}/raw/gdelt",
    "--manifest-prefix",
    f"runs/{RUN_ID}/manifests/gdelt",
    "--workers",
    os.environ.get("MP_GDELT_WORKERS", "8"),
    "--resume",
]
print(" ".join(str(x) for x in full_cmd))
run(full_cmd)

In [ ]:
FULL_MANIFEST_FILE = RUN_DIR / "gdelt_full_manifest.jsonl"
full_manifest_key = f"runs/{RUN_ID}/manifests/gdelt/run_manifest.jsonl"
s3.download_file(RUN_BUCKET, full_manifest_key, str(FULL_MANIFEST_FILE))
full_manifest = pd.read_json(FULL_MANIFEST_FILE, lines=True)
assert len(full_manifest) == len(plan)
assert full_manifest["stamp"].nunique() == len(plan)
assert full_manifest["status"].isin(["success", "empty"]).all()
assert full_manifest["md5_ok"].all()
assert (full_manifest["parsed_rows"] > 0).all()
full_manifest.to_csv(RUN_DIR / "gdelt_full_manifest.csv", index=False)
print(full_manifest["status"].value_counts())
print(full_manifest[["parsed_rows", "matched_rows"]].sum())

## 12. EMR Spark stages

不要用旧 launcher。由 notebook 创建无 SSH key 的 auto-terminating EMR cluster。Jerry 先在 shell 里设置：

```bash
export MP_ALLOW_EMR=1
export MP_EMR_RELEASE=emr-7.14.0
export MP_EMR_SUBNET_ID=subnet-xxxxxxxx
export MP_EMR_LOG_URI=s3://new-versioned-run-bucket/logs/
export MP_EMR_SERVICE_ROLE=EMR_DefaultRole_V2
export MP_EMR_JOB_FLOW_ROLE=EMR_EC2_DefaultRole
export MP_EMR_MASTER_TYPE=m6i.xlarge
export MP_EMR_CORE_TYPE=r6i.2xlarge
export MP_EMR_CORE_COUNT=4
```

instance types、roles 和 subnet 必须由 Jerry 按自己的 AWS account 实际值填写。subnet 应是 private subnet；不要创建对 `0.0.0.0/0` 的 SSH rule。先核对所选 EMR release 的 Spark version，并把 release label 写进论文复现材料。

In [ ]:
assert ALLOW_EMR
EMR_RELEASE = os.environ.get("MP_EMR_RELEASE", "emr-7.14.0")
SUBNET_ID = os.environ["MP_EMR_SUBNET_ID"]
LOG_URI = os.environ["MP_EMR_LOG_URI"]
SERVICE_ROLE = os.environ["MP_EMR_SERVICE_ROLE"]
JOB_FLOW_ROLE = os.environ["MP_EMR_JOB_FLOW_ROLE"]
MASTER_TYPE = os.environ.get("MP_EMR_MASTER_TYPE", "m6i.xlarge")
CORE_TYPE = os.environ.get("MP_EMR_CORE_TYPE", "r6i.2xlarge")
CORE_COUNT = int(os.environ.get("MP_EMR_CORE_COUNT", "4"))
emr = session.client("emr")
labels = emr.list_release_labels(MaxResults=100)["ReleaseLabels"]
assert EMR_RELEASE in labels
assert LOG_URI.startswith("s3://")
print(EMR_RELEASE, MASTER_TYPE, CORE_TYPE, CORE_COUNT)

In [ ]:
code_prefix = f"s3://{RUN_BUCKET}/runs/{RUN_ID}/code"
data_prefix = f"s3://{RUN_BUCKET}/runs/{RUN_ID}"
spark_steps = [
    {
        "Name": "tfidf",
        "ActionOnFailure": "TERMINATE_CLUSTER",
        "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": [
                "spark-submit",
                "--deploy-mode",
                "cluster",
                f"{code_prefix}/08_spark_preprocess_tfidf.py",
                "--fomc-path",
                f"{data_prefix}/raw/fomc/*/*.json",
                "--nyt-path",
                f"{data_prefix}/raw/nyt/*/*.json",
                "--guardian-path",
                f"{data_prefix}/raw/guardian/*/*.json",
                "--gdelt-path",
                f"{data_prefix}/raw/gdelt/",
                "--tokens-out",
                f"{data_prefix}/curated/tokens",
                "--tfidf-out",
                f"{data_prefix}/curated/tfidf",
                "--model-out",
                f"{data_prefix}/models/tfidf",
                "--fit-start",
                "2015-03-15",
                "--fit-cutoff",
                "2019-03-23",
            ],
        },
    },
    {
        "Name": "meeting-windows",
        "ActionOnFailure": "TERMINATE_CLUSTER",
        "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": [
                "spark-submit",
                "--deploy-mode",
                "cluster",
                f"{code_prefix}/09_spark_meeting_windows.py",
                "--tfidf",
                f"{data_prefix}/curated/tfidf",
                "--meetings",
                f"{data_prefix}/ref/meetings_all88.csv",
                "--output",
                f"{data_prefix}/curated/meeting_windows",
                "--max-lag",
                str(PAD_DAYS),
            ],
        },
    },
    {
        "Name": "similarity",
        "ActionOnFailure": "TERMINATE_CLUSTER",
        "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": [
                "spark-submit",
                "--deploy-mode",
                "cluster",
                f"{code_prefix}/11_spark_similarity.py",
                "--windows",
                f"{data_prefix}/curated/meeting_windows",
                "--tfidf",
                f"{data_prefix}/curated/tfidf",
                "--output",
                f"{data_prefix}/processed/per_meeting",
            ],
        },
    },
]
print(json.dumps(spark_steps, indent=2))

In [ ]:
cluster = emr.run_job_flow(
    Name=f"makkapakka-{RUN_ID}",
    ReleaseLabel=EMR_RELEASE,
    Applications=[{"Name": "Hadoop"}, {"Name": "Spark"}],
    LogUri=LOG_URI,
    ServiceRole=SERVICE_ROLE,
    JobFlowRole=JOB_FLOW_ROLE,
    VisibleToAllUsers=True,
    Instances={
        "Ec2SubnetId": SUBNET_ID,
        "KeepJobFlowAliveWhenNoSteps": False,
        "InstanceGroups": [
            {"Name": "Master", "Market": "ON_DEMAND", "InstanceRole": "MASTER", "InstanceType": MASTER_TYPE, "InstanceCount": 1},
            {"Name": "Core", "Market": "ON_DEMAND", "InstanceRole": "CORE", "InstanceType": CORE_TYPE, "InstanceCount": CORE_COUNT},
        ],
    },
    Steps=spark_steps,
    AutoTerminationPolicy={"IdleTimeout": 3600},
    Tags=[{"Key": "project", "Value": "makkapakka"}, {"Key": "run_id", "Value": RUN_ID}],
)
CLUSTER_ID = cluster["JobFlowId"]
(RUN_DIR / "emr_cluster_id.txt").write_text(CLUSTER_ID, encoding="utf-8")
print(CLUSTER_ID)

下面的 status cell 手动重跑，不要写无休止 polling loop。只有三个 steps 全为 COMPLETED 才下载结果。失败时先看 S3 logs，修代码后使用新的 run suffix；不要覆盖一个已部分完成的 processed prefix。

In [ ]:
if "CLUSTER_ID" not in globals():
    CLUSTER_ID = (RUN_DIR / "emr_cluster_id.txt").read_text(encoding="utf-8").strip()
cluster_state = emr.describe_cluster(ClusterId=CLUSTER_ID)["Cluster"]
step_states = emr.list_steps(ClusterId=CLUSTER_ID)["Steps"]
print(cluster_state["Status"]["State"])
for step in reversed(step_states):
    print(step["Name"], step["Status"]["State"])
with open(RUN_DIR / "emr_cluster_description.json", "w", encoding="utf-8") as handle:
    json.dump(cluster_state, handle, indent=2, default=str)

## 13. 下载结果并做 hard QC

不要先覆盖 `data_reference/per_meeting.parquet`。先下载到 versioned handoff directory，完成 QC 和双人 sign-off 后才改变 downstream input。

In [ ]:
assert all(step["Status"]["State"] == "COMPLETED" for step in step_states)
DELIVERY_DIR = RUN_DIR / "per_meeting_2015_2025"
DELIVERY_DIR.mkdir(parents=True, exist_ok=True)
run([
    "aws",
    "s3",
    "sync",
    f"s3://{RUN_BUCKET}/runs/{RUN_ID}/processed/per_meeting/",
    DELIVERY_DIR,
    "--only-show-errors",
])
delivered = pd.read_parquet(DELIVERY_DIR)
print(len(delivered))
print(delivered.columns.tolist())

In [ ]:
required = {
    "meeting_date",
    "lag_days",
    "source",
    "corpus",
    "weighting",
    "cosine_sim",
    "n_docs",
    "n_urls",
    "mean_native_tone",
    "model_fit_cutoff",
    "n_domains",
    "representation_type",
}
assert required.issubset(delivered.columns)
delivered["meeting_date"] = pd.to_datetime(delivered["meeting_date"])
keys = ["meeting_date", "lag_days", "source", "corpus", "weighting"]
assert not delivered.duplicated(keys).any()
assert np.isfinite(delivered["cosine_sim"].dropna()).all()
assert delivered["cosine_sim"].dropna().between(0, 1).all()
assert (delivered["n_docs"].dropna() > 0).all()
assert (delivered["n_domains"].dropna() > 0).all()

primary = delivered[
    (delivered["source"] == "gdelt")
    & (delivered["corpus"] == "direct_fed")
    & (delivered["weighting"] == "domain_equal")
]
lag_m2 = primary[primary["lag_days"] == -2]
available_meetings = set(coverage.loc[coverage["lag_m2_available"], "meeting_date"])
observed_meetings = set(lag_m2.loc[lag_m2["cosine_sim"].notna(), "meeting_date"])
assert observed_meetings.issubset(available_meetings)
assert len(observed_meetings) <= 86
pd.DataFrame({
    "meeting_date": sorted(available_meetings),
    "primary_lag_m2_observed": [d in observed_meetings for d in sorted(available_meetings)],
}).to_csv(RUN_DIR / "primary_lag_m2_coverage.csv", index=False)
print("archive-available", len(available_meetings), "primary-observed", len(observed_meetings))
print(delivered.groupby(["source", "corpus", "weighting"])["meeting_date"].nunique())
print(primary.groupby("lag_days")["meeting_date"].nunique())
print(primary[["n_docs", "n_domains"]].describe())

人工 QC 还要完成：

- 按年份、direct/broad、domain 分层抽至少 200 条记录，两位 coder 独立判断是否真正涉及 Fed monetary policy。报告 precision、agreement、Cohen's kappa 和 adjudication rule。
- 报告 unique document、unique URL、duplicate ID、duplicate URL、top-domain shares、effective number of domains。
- 对 article-weighted 与 domain-equal-weighted 的 similarity 做相关和差异表。
- 若无法取得 article full text，全文统一改成 GKG metadata alignment，不能再写 article-language similarity 或 population of Fed news。
- 若一定要保留 article-text claim，必须另取可授权的 full-text subset，并做 metadata measure 的 convergence validation。

## 14. Jerry 交付给 Doris 的文件

Jerry 不要只发一个 parquet。完整交付应包含：

1. versioned `per_meeting_2015_2025` parquet directory
2. `gdelt_download_plan.csv` 与 metadata JSON
3. full shard manifest，包含 success/empty/failed、MD5 和 row counts
4. patched source code 与 code hashes
5. fitted TF-IDF pipeline model、fit cutoff 和 parameters
6. source/year/domain/document coverage tables
7. duplicate and syndication audit
8. manual corpus-validation coding sheet
9. EMR release、Spark version、cluster description、step states 和 log URI
10. `pip_freeze.txt`
11. exact S3 run prefix 和 final artifact hashes

没有这些材料，不能在稿件中继续声称 fully reproducible。

## 15. 数据回来后的本地分析顺序

不要立刻把新 parquet 覆盖到 `data_reference`。先把下游脚本改为接受显式 `--per-meeting --market-dir --output-dir --outcome`。使用新的 results directory。顺序如下：

1. 重新构造严格 market outcomes 和 panel。
2. 在完全没调参的 holdout 53 meetings 上单独估计 primary specification。
3. 再给 discovery 33 和 pooled 86。
4. 做 direct/broad、article/domain weighting、strict/event outcome robustness。
5. 对 -7 到 +7 做 max-|t| correction。
6. 用 moving-block/circular permutation 和 time-aware bootstrap。
7. 做 weekday/month-matched non-FOMC pseudo-event placebo。
8. MPS 用 consistent USMPD；若暂时没有完整 series，就分开 full sample 与 restricted sample。
9. 删除或降级 n 很小且缺诊断的 VAR/Granger；若保留，补 stationarity、stability、residual tests 和 IRF confidence intervals。
10. 所有 tables/figures 从新 results 自动生成，再更新 manuscript。

In [ ]:
ANALYSIS_READY = os.environ.get("MP_ANALYSIS_READY", "0") == "1"
assert ANALYSIS_READY
ANALYSIS_OUT = RUN_DIR / "analysis_results"
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

analysis_commands = [
    [sys.executable, UPDATE / "scripts" / "analysis" / "00_build_panel.py",
     "--per-meeting", DELIVERY_DIR, "--market-dir", SPX_FILE.parent, "--output-dir", ANALYSIS_OUT],
    [sys.executable, UPDATE / "scripts" / "analysis" / "12_main_regression.py",
     "--input-dir", ANALYSIS_OUT, "--output-dir", ANALYSIS_OUT, "--sample", "holdout"],
    [sys.executable, UPDATE / "scripts" / "analysis" / "13_identification.py",
     "--input-dir", ANALYSIS_OUT, "--output-dir", ANALYSIS_OUT, "--sample", "holdout"],
]
for command in analysis_commands:
    print(" ".join(str(x) for x in command))
    run(command, cwd=UPDATE)


三个入口已经接到版本化面板和 temporal-validation 实现。主规格先保存 holdout 结果，再运行 discovery 与 pooled，具体命令和输出结构见 [分析说明](../scripts/analysis/README.md)。

MPS 定义与 Jerry 讨论后确定。当前面板命令只接入 SPX 和 VIX；MPS 附加规格要求显式传入冻结的 surprise CSV 和定义。非 FOMC placebo 所需的匹配日期会写入 panel/pseudo_event_requests.csv，后续需要用同一冻结模型计算这些日期的相似度。

每个阶段以目录原子发布并记录输入、协议和代码 hash。使用新的 ANALYSIS_OUT 保存新的 run。表图和论文在正式验证结果保存之后更新。


## 16. 是否只剩收尾

不是。Jerry 的云端 run 解决的是 2019 后 GDELT 缺失和样本太小，但不会自动解决以下问题：

- outcome 构造错误
- metadata 被写成 article text 的 construct-validity 问题
- lag -2 data-driven selection 与 multiple testing
- minutes 在会后数周发布，因此 media-to-eventual-minutes similarity 不是实时可用 forecast
- MPS 截止 2023 造成的 sample-selection 问题
- broad filter 的 false positives、duplicate/syndication 和 domain concentration
- curated sources 与 GDELT 不同字段、不同 coverage，不能直接归因为 breadth
- non-FOMC placebo、time-aware inference、holdout validation 和 equivalence testing 缺失
- manuscript 中硬编码的 n、date、coefficient、caption、abstract、title 和 causal wording
- Makefile/Snakefile、environment lock、data dictionary、license、DOI、double-blind version

真正可以进入投稿收尾的 go/no-go 条件：

1. 86-meeting lag -2 coverage 与 archive gaps 被完整记录。
2. strict outcome 与 event-inclusive robustness 都通过日期级 QC。
3. untouched 53-meeting holdout 结果先于任何重新调参被保存。
4. direct-Fed corpus 经过双人 validation，metadata 表述被全稿统一。
5. multiple-testing、block permutation、bootstrap、pseudo-events 完成。
6. full-sample 与 MPS-restricted regressions 分开。
7. 新结果无论显著与否都被诚实写入，不用旧 0.565 headline 覆盖。
8. tables、figures、paper、appendix 和 Word/PDF 全部由同一次 frozen run 生成。
9. 独立复现从空 results directory 成功。
10. 匿名化、期刊格式、cover letter、data/code availability 最后完成。

若 holdout 不支持旧发现，论文仍可改成严谨的 temporal-validation/null-result paper，但标题、摘要和贡献必须重写，不能继续使用 `Anticipation, Not Reaction` 的强因果暗示。